# SED-39: retrain with the FSK/PSK/QAM data fixed

**Why.** `make_fskpsk_shards.py` generated its 18 classes (2/4/8FSK, GFSK, GMSK, MSK, CPFSK, BPSK, QPSK, OQPSK, 8/16/32PSK, 16-256QAM) with no impairments, so every one of those training windows sat exactly at 0 Hz, while every other family was placed anywhere in the band. The model learned the position instead of the waveform: BPSK is right 100% at 0 Hz and 4% at +300 kHz (called DSSS), and DSSS is called BPSK whenever it is centred. In the pipeline DSSS can never be right, and real comms signals with a frequency offset fall apart.

**What this notebook does.**
* **Part A, data (no GPU needed, about 30-45 min):** copies the shards, re-makes the generated FSK/PSK/QAM rows from their stored seeds *with* the impairment chain (`scripts/reimpair_fskpsk_shards.py`; RadioML/HisarMod rows are kept, the merge already places them), merges a new combined set, checks it and saves it to Drive. Skipped automatically when it is already on Drive.
* **Part B, training (T4 GPU, about 2 h):** the spectrum model's exact settings (seed 4, n_fft 64, expert features, `--full-spectrum`, 15 epochs) on the new data, then calibration, then a fair comparison: **both** models scored on **both** test sets.

**Short on GPU time?** Run Part A on a CPU runtime first (Runtime -> Change runtime type -> CPU), then switch to T4 and run everything again from the top: Part A sees the data on Drive and skips.

**If Colab disconnects:** run every cell again from the top. Part A skips what is done; training resumes from the last saved epoch.

## 1. Settings

In [ ]:
SEED = 4
BRANCH = 'eileenyeoh06/sed-39-classifier-experiments-to-raise-accuracy'
FOI26 = '/content/drive/MyDrive/sedic/FOI26'      # the shared folder (add a shortcut to My Drive if it was shared with you)

SHARDS_ON_DRIVE = f'{FOI26}/shards_data'                     # analog/ apsk/ fskpsk/ jam/ ... as the team made them
DATA_OUT = f'{FOI26}/combined_data_labels_v2_fskpskfix'       # the new combined set (Part A writes it, Part B reads it)
OLD_DATA = f'{FOI26}/combined_data_labels_v2'                 # a5cda92d476f, for the fair comparison

RUN_NAME = '2026-11-07_nfft64_ep15_labelsv2fix_expert_spectrum'   # typed by hand so a rerun tomorrow resumes
EPOCHS = 15
XF = '--expert-features --full-spectrum'
# Physics branch (docs/SEPARABILITY_ANALYSIS.md): True trains the same model plus the 56 physics
# numbers (src/physics_features.py). Needs branch eavan-accuracy-plan on GitHub. Own run name,
# so it never resumes or overwrites the run without it.
PHYSICS = False
if PHYSICS:
    BRANCH = 'eavan-accuracy-plan'
    XF += ' --physics-features'
    RUN_NAME += '_physics'
CKPT_DIR = f'{FOI26}/results/{RUN_NAME}_seed{SEED}'
OLD_MODEL = f'{FOI26}/results/2026-11-06_nfft64_ep15_labelsv2_expert_spectrum_seed4/best.pt'   # the 52% model (epoch 17)

## 2. Mount Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf sedicAI_NEXA
!git clone -q -b {BRANCH} https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!git log -1 --format='commit %H  %s'
!pip install -q pyyaml
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (fine for Part A, needed for Part B)')

# Part A: rebuild the data

## 3. Copy the shards to the Colab disk (a few minutes)

In [ ]:
import os, json, glob
PART_A_DONE = os.path.exists(f'{DATA_OUT}/combined.json')
GROUPS = ['analog', 'apsk', 'fskpsk', 'jam', 'multicarrier', 'pulse', 'radar', 'spread']
if PART_A_DONE:
    print('New data already on Drive:', json.load(open(f'{DATA_OUT}/combined.json'))['combined_version'], '-> skipping Part A')
else:
    !mkdir -p /content/shards
    for g in GROUPS:
        !cp -r "{SHARDS_ON_DRIVE}/{g}" /content/shards/
    for g in GROUPS:
        print(g, len(glob.glob(f'/content/shards/{g}/*.npz')), 'shards')

## 4. Re-make the generated FSK/PSK/QAM rows with impairments (about 10 min)

Every generated row is re-made from its stored seed; spot checks first confirm the seed still reproduces the stored window, so labels and SNRs cannot drift. External rows are copied unchanged.

In [ ]:
if not PART_A_DONE:
    # Manifests for the merge: every group's as committed, with fskpsk replaced by the new ones.
    !rm -rf /content/manifests /content/fskpsk_new /content/fskpsk_new_manifests
    !mkdir -p /content/manifests
    !cp data/manifests/*.json /content/manifests/
    !rm -f /content/manifests/fskpsk_*.json /content/manifests/mix_*.json
    !python scripts/reimpair_fskpsk_shards.py --shards /content/shards --manifests data/manifests --out /content/fskpsk_new --manifests-out /content/fskpsk_new_manifests
    !rm -rf /content/shards/fskpsk && mv /content/fskpsk_new /content/shards/fskpsk
    !cp /content/fskpsk_new_manifests/*.json /content/manifests/
    print(len(glob.glob('/content/manifests/*.json')), 'manifests for the merge')

## 5. Merge into a new combined set (about 20-30 min)

Same merge as `a5cda92d476f` (external fix + labels-v2, mixtures, seed 4100); only the FSK/PSK/QAM shards changed, so the counts must match the old set exactly.

In [ ]:
if not PART_A_DONE:
    !rm -rf /content/combined /content/mix_manifests
    !python scripts/merge_shards.py --manifests /content/manifests --shards /content/shards --out /content/combined --mix-out /content/shards/mix --mix-manifests /content/mix_manifests

## 6. Check the new set, then save it to Drive

In [ ]:
import sys
import numpy as np
sys.path.insert(0, '.')
from src.config import DATASET_CLASSES as C
if not PART_A_DONE:
    for split, n in (('train', 960_000), ('val', 120_000), ('test', 120_000)):
        X = np.load(f'/content/combined/{split}/X.npy', mmap_mode='r'); y = np.load(f'/content/combined/{split}/y.npy', mmap_mode='r')
        print(split, X.shape, y.shape)
        assert X.shape == (n, 2, 512) and y.shape == (n, 60), f'{split} has a different size from a5cda92d476f: stop and tell Eileen'
    # Are the FSK/PSK/QAM windows off 0 Hz now? (Before the fix almost all were at 0 Hz.)
    X = np.load('/content/combined/test/X.npy', mmap_mode='r'); y = np.load('/content/combined/test/y.npy'); snr = np.load('/content/combined/test/snr_labels.npy')
    f = np.fft.fftshift(np.fft.fftfreq(512, 1 / 3.2e6))
    for cls in ('BPSK', 'GFSK', '16QAM'):
        rows = np.flatnonzero((y[:, C.index(cls)] == 1) & (y.sum(1) == 1) & (snr >= 10))[:500]
        z = X[rows, 0].astype(np.float32) + 1j * X[rows, 1].astype(np.float32)
        P = np.abs(np.fft.fftshift(np.fft.fft(z, axis=1), axes=1)) ** 2
        centre = (P * f).sum(1) / P.sum(1)
        print(f'{cls:6s} clean test windows within 20 kHz of 0 Hz: {np.mean(np.abs(centre) < 20e3):.0%}')
    print('saving to Drive (~2.5 GB) ...')
    !mkdir -p "{DATA_OUT}"
    !cp -r /content/combined/train /content/combined/val /content/combined/test "{DATA_OUT}/"
    !cp /content/combined/combined.json "{DATA_OUT}/"
    print('saved:', json.load(open('/content/combined/combined.json'))['combined_version'])

# Part B: train (T4 GPU)

## 7. Copy the new data to the Colab disk

In [ ]:
import torch
assert torch.cuda.is_available(), 'Part B needs a GPU: Runtime -> Change runtime type -> T4 GPU, then run all cells again (Part A will skip)'
if not os.path.exists('/content/combined/combined.json'):
    !mkdir -p /content/combined
    !cp -r "{DATA_OUT}/train" "{DATA_OUT}/val" "{DATA_OUT}/test" /content/combined/
    !cp "{DATA_OUT}/combined.json" /content/combined/
NEW_VERSION = json.load(open('/content/combined/combined.json'))['combined_version']
print('training on', NEW_VERSION, '(the old set was a5cda92d476f)')

## 8. Quick check (about 1 minute)

In [ ]:
!python scripts/train_week1.py --seed {SEED} --data /content/combined --limit 2000 --epochs 1 --batch-size 256 {XF} --ckpt-dir /content/try --fresh

## 9. Train (about 2 h on a T4)

One line per epoch; every epoch is saved to `CKPT_DIR` on Drive. At the end it scores the new test split (`scorecard.json`).

In [ ]:
!python scripts/train_week1.py --seed {SEED} --data /content/combined --epochs {EPOCHS} --batch-size 256 {XF} --ckpt-dir "{CKPT_DIR}" --eval-test

## 10. Calibrate on val (about 2 minutes)

In [ ]:
!python scripts/calibrate_week1.py --ckpt "{CKPT_DIR}/best.pt" --data /content/combined --cache /content/logits_new

## 11. Fair comparison: both models on both test sets

The new test set's FSK/PSK/QAM windows are off-centre, so the two scorecards alone are not comparable. This scores the old model and the new one on the old test set (FSK/PSK/QAM all centred) and on the new one (placed like every other family): overall, at >= 10 dB and per family. Writes `fair_compare.json` next to the new model.

In [ ]:
from src.config import CLASS_GROUPS
from src.models.amc_cnn import AMC_CNN
dev = torch.device('cuda')
FAM = {c: g for g, cs in CLASS_GROUPS.items() for c in cs}; FAM['NOISE_FLOOR'] = 'NOISE'

def load(path):
    ck = torch.load(path, map_location=dev, weights_only=False); m = ck['meta']
    net = AMC_CNN(num_classes=60, input_len=m['input_len'], stft_n_fft=m['stft_n_fft'], **m.get('model_flags', {})).to(dev)
    net.load_state_dict(ck['state_dict'])
    return net.eval()

@torch.no_grad()
def top_of(net, X):
    return np.concatenate([net(torch.from_numpy(np.asarray(X[s:s + 2048], dtype=np.float32)).to(dev)).argmax(1).cpu().numpy()
                           for s in range(0, len(X), 2048)])

if not os.path.exists('/content/old_test/X.npy'):
    !mkdir -p /content/old_test
    !cp "{OLD_DATA}/test/X.npy" "{OLD_DATA}/test/y.npy" "{OLD_DATA}/test/snr_labels.npy" /content/old_test/
sets = {'old test': '/content/old_test', 'new test': '/content/combined/test'}
models = {'old model': OLD_MODEL, 'new model': f'{CKPT_DIR}/best.pt'}
fams = sorted(set(FAM.values()))
res = {}
for sname, d in sets.items():
    X = np.load(f'{d}/X.npy', mmap_mode='r'); y = np.load(f'{d}/y.npy').astype(bool); snr = np.load(f'{d}/snr_labels.npy')
    tf = np.array([FAM[C[i]] for i in y.argmax(1)]); single = y.sum(1) == 1
    for mname, path in models.items():
        top = top_of(load(path), X)
        hit = y[np.arange(len(y)), top]
        res[f'{mname} / {sname}'] = {'top1': float(hit.mean()), 'top1_high': float(hit[snr >= 10].mean()),
                                     'family': {g: float(hit[single & (tf == g)].mean()) for g in fams}}
cols = list(res)
print(f"{'':14s}" + ''.join(f'{c:>22s}' for c in cols))
print(f"{'top-1':14s}" + ''.join(f"{res[c]['top1']:22.3f}" for c in cols))
print(f"{'top-1 >=10 dB':14s}" + ''.join(f"{res[c]['top1_high']:22.3f}" for c in cols))
for g in fams:
    print(f'{g:14s}' + ''.join(f"{res[c]['family'][g]:22.3f}" for c in cols))
json.dump(res, open(f'{CKPT_DIR}/fair_compare.json', 'w'), indent=2)
print('\nwrote', f'{CKPT_DIR}/fair_compare.json')

## 12. The position check: the same signal at 0 Hz and at +300 kHz

Before the fix: BPSK 100% / 4%, DSSS 0% / 96%. After it, the two columns should be close for every class. **Tell Eileen when this is done**; she can read everything from Drive.

In [ ]:
from src.data import impairments
from src.data.merge import normalize_batch
from src.generators import digital, spread
rng = np.random.default_rng(0)
nets = {m: load(p) for m, p in models.items()}
print(f"{'class':8s}" + ''.join(f'{m + " @ " + o:>22s}' for m in nets for o in ('0 Hz', '+300 kHz')))
for gen, cls in [(digital, c) for c in ('BPSK', 'QPSK', 'GFSK', '2FSK', '16QAM')] + [(spread, 'DSSS')]:
    row = []
    for net in nets.values():
        for off in (0.0, 300e3):
            W = []
            for _ in range(200):
                x, _ = gen.generate_window(cls, None, rng)
                x = np.asarray(x, complex); x /= np.sqrt(np.mean(np.abs(x) ** 2))
                W.append(impairments.frequency_shift(x, off) + impairments.noise(512, 10 ** -1.5, rng))
            with torch.no_grad():
                top = net(torch.from_numpy(normalize_batch(np.stack(W))).to(dev)).argmax(1).cpu().numpy()
            row.append((top == C.index(cls)).mean())
    print(f'{cls:8s}' + ''.join(f'{v:22.2f}' for v in row))